# 02 — Baseline Model (TF-IDF + Cosine Similarity)

This notebook implements a lightweight baseline to score Resume-JD similarity using TF-IDF vectors and cosine similarity.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_recall_curve, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

## Load processed dataset

In [2]:
data_path = Path('data/processed/resume_jd_clean.csv')
if not data_path.exists():
    raise FileNotFoundError('Run 01_EDA_and_Preprocessing.ipynb first to create processed data.')

df = pd.read_csv(data_path)
df.head(3)

,resume_text,jd_text,label,sample_id
0,summaryhighly motivated sales associate with e...,net2source inc. is an award-winning total work...,0,0
1,professional summarycurrently working with cat...,at salas obrien we tell our clients that were ...,0,1
2,summaryi started my construction career in jun...,schweitzer engineering laboratories sel infras...,0,2


## Train/Test split

In [3]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['label'],
)

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

Train size: 4992
Test size: 1248


## Fit TF-IDF and compute pairwise cosine scores

A shared vectorizer is fit on both resume and JD corpora from the training split.

In [4]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
)

train_corpus = pd.concat([train_df['resume_text'], train_df['jd_text']], axis=0).tolist()
vectorizer.fit(train_corpus)

X_train_resume = vectorizer.transform(train_df['resume_text'])
X_train_jd = vectorizer.transform(train_df['jd_text'])
train_scores = cosine_similarity(X_train_resume, X_train_jd).diagonal()

X_test_resume = vectorizer.transform(test_df['resume_text'])
X_test_jd = vectorizer.transform(test_df['jd_text'])
test_scores = cosine_similarity(X_test_resume, X_test_jd).diagonal()

## Choose threshold on train set (maximize F1)

In [5]:
y_train = train_df['label'].to_numpy()
precision, recall, thresholds = precision_recall_curve(y_train, train_scores)
f1_curve = (2 * precision * recall) / np.clip(precision + recall, 1e-12, None)
best_idx = np.nanargmax(f1_curve[:-1])
best_threshold = float(thresholds[best_idx])

print(f"Selected threshold: {best_threshold:.4f}")
print(f"Best train F1: {f1_curve[best_idx]:.4f}")

Selected threshold: 0.0045
Best train F1: 0.6648


## Evaluate baseline on test set

In [6]:
y_test = test_df['label'].to_numpy()
y_pred = (test_scores >= best_threshold).astype(int)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, test_scores):.4f}")
print('Classification report:')
print(classification_report(y_test, y_pred, digits=4))

Accuracy: 0.5008
F1: 0.6638
ROC-AUC: 0.6246
Classification report:
              precision    recall  f1-score   support

           0     0.6667    0.0159    0.0311       628
           1     0.4988    0.9919    0.6638       620

    accuracy                         0.5008      1248
   macro avg     0.5827    0.5039    0.3474      1248
weighted avg     0.5833    0.5008    0.3454      1248



## Baseline ranking demo (single JD, many resumes)

In [7]:
sample_jd = test_df.iloc[0]['jd_text']
candidate_resumes = test_df['resume_text'].head(10).tolist()

jd_vec = vectorizer.transform([sample_jd])
resume_vecs = vectorizer.transform(candidate_resumes)
rank_scores = cosine_similarity(resume_vecs, jd_vec).ravel()

ranked = pd.DataFrame({
    'resume_idx': list(range(len(candidate_resumes))),
    'score': rank_scores,
}).sort_values('score', ascending=False).reset_index(drop=True)

ranked

,resume_idx,score
0,8,0.046265
1,3,0.033827
2,2,0.025574
3,5,0.023667
4,4,0.013392
5,0,0.011453
6,6,0.011334
7,1,0.009251
8,9,0.007550
9,7,0.006774
